# Prepare Data
Scans the dermoscopy dataset and creates `dataset_manifest.csv` with patient-level stratified 5-fold splits.

**Folder structure expected:**
```
dermoscopy/
    DN/
        <patient_folder>/
            image1.jpg
            image2.jpg
    MIA/
        <patient_folder>/
            ...
    Minsitu/
        <patient_folder>/
            ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
from pathlib import Path

# ── Edit these paths to match your Drive layout ────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/PanDerm_Scripts')
DATA_ROOT   = DRIVE_ROOT / 'dermoscopy'       # folder with DN / MIA / Minsitu subfolders
OUTPUT_DIR  = DRIVE_ROOT / 'results'          # where dataset_manifest.csv will be saved
# ───────────────────────────────────────────────────────────────────────────────

CLASS_NAMES  = ['DN', 'MIA', 'Minsitu']
CLASS_LABELS = {'DN': 0, 'MIA': 1, 'Minsitu': 2}
IMAGE_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
N_FOLDS      = 5
RANDOM_SEED  = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config ready')
print(f'  Data root  : {DATA_ROOT}')
print(f'  Output dir : {OUTPUT_DIR}')

## Cell 3 — Scan dataset

In [ ]:
import pandas as pd

rows = []
for class_name in CLASS_NAMES:
    label     = CLASS_LABELS[class_name]
    class_dir = DATA_ROOT / class_name
    if not class_dir.exists():
        print(f'  WARNING: class directory not found: {class_dir}')
        continue

    for patient_dir in sorted(class_dir.iterdir()):
        if not patient_dir.is_dir():
            continue

        folder_name = patient_dir.name
        patient_id  = folder_name.split()[0]   # numeric ID before any spaces

        images = [f for f in patient_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
        if not images:
            print(f'  Skipping empty folder: {class_name}/{folder_name}')
            continue

        for img_path in sorted(images):
            rows.append({
                'image_path':  str(img_path),
                'patient_id':  patient_id,
                'folder_name': folder_name,
                'diagnosis':   class_name,
                'label':       label,
            })

df = pd.DataFrame(rows)
print(f'Found {len(df)} images from {df["patient_id"].nunique()} patients')
print(df['diagnosis'].value_counts())

## Cell 4 — Create stratified k-fold splits

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Per-patient majority label for stratification
patient_label = (
    df.groupby('patient_id')['label']
    .agg(lambda x: x.value_counts().index[0])
)

groups       = df['patient_id'].values
strat_labels = df['patient_id'].map(patient_label).values

df['fold'] = -1
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

for fold_idx, (_, test_idx) in enumerate(sgkf.split(df, strat_labels, groups)):
    df.loc[df.index[test_idx], 'fold'] = fold_idx

assert (df['fold'] == -1).sum() == 0, 'Some images were not assigned a fold!'
print('Fold assignment complete')
print(df['fold'].value_counts().sort_index())

## Cell 5 — Validate no data leakage

In [ ]:
ok = True
for fold_idx in range(N_FOLDS):
    test_patients  = set(df.loc[df['fold'] == fold_idx,  'patient_id'])
    train_patients = set(df.loc[df['fold'] != fold_idx, 'patient_id'])
    overlap = test_patients & train_patients
    if overlap:
        print(f'  LEAK in fold {fold_idx}: {overlap}')
        ok = False

if ok:
    print('OK: No patient leakage detected across all folds')

## Cell 6 — Save manifest + print statistics

In [ ]:
csv_path = OUTPUT_DIR / 'dataset_manifest.csv'
df.to_csv(csv_path, index=False)
print(f'Manifest saved: {csv_path}')
print(f'  Total images  : {len(df)}')
print(f'  Total patients: {df["patient_id"].nunique()}')

# Per-class stats
print(f"\n{'Class':<12} {'Patients':>10} {'Images':>10}")
print('-' * 35)
for cls in CLASS_NAMES:
    sub   = df[df['diagnosis'] == cls]
    n_pat = sub['patient_id'].nunique()
    n_img = len(sub)
    print(f'{cls:<12} {n_pat:>10} {n_img:>10}')

# Multi-class patients
pat_classes = df.groupby('patient_id')['diagnosis'].nunique()
multi = pat_classes[pat_classes > 1]
if len(multi) > 0:
    print(f'\nMulti-class patients ({len(multi)}):')
    for pid in multi.index:
        classes = df.loc[df['patient_id'] == pid, 'diagnosis'].unique()
        print(f'  {pid}: {", ".join(classes)}')

# Per-fold breakdown
print(f"\n{'Fold':<6} {'DN_img':>8} {'MIA_img':>8} {'Minsitu_img':>12} {'Total':>8}")
print('-' * 46)
for fold_idx in range(N_FOLDS):
    test = df[df['fold'] == fold_idx]
    dn  = len(test[test['diagnosis'] == 'DN'])
    mia = len(test[test['diagnosis'] == 'MIA'])
    mis = len(test[test['diagnosis'] == 'Minsitu'])
    print(f'{fold_idx:<6} {dn:>8} {mia:>8} {mis:>12} {len(test):>8}')